# 03 WeightWatcher Analysis

Manifest-driven WeightWatcher validation and publication-figure export.

In [ ]:
# User-editable configuration: keep path/settings changes in this cell.
from pathlib import Path
import os

DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/tmp/wwpgd_v2/data"))
RESULTS_ROOT_SETTING = os.environ.get(
    "WWGPT_RESULTS_ROOT",
    os.environ.get("RESULTS_ROOT", "/tmp/wwpgd_v2/real_level0_results_v5"),
)
RUN_LOG = Path(os.environ.get("RUN_LOG", "/tmp/wwpgd_v2/real_level0_five_seed_v4.log"))
PID_FILE = Path(os.environ.get("PID_FILE", "/tmp/wwpgd_v2/real_level0_five_seed_v4.pid"))
INCLUDE_LEGACY = False
PLOT_BAND = "mean_std"  # "mean_std" or "mean_ci95" across independent seeds
PUBLICATION_DPI = 300
SOURCE_DATA_FORMAT = "csv"

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
repo = cwd if (cwd / "src" / "wwgpt").exists() else cwd.parent
if str(repo / "src") not in sys.path:
    sys.path.insert(0, str(repo / "src"))

from wwgpt.analysis import (
    audit_spectral_validity,
    build_run_inventory,
    discover_canonical_runs,
    discover_pair_candidates,
    load_run_artifacts,
    normalize_metrics,
    normalize_projection_records,
    normalize_spectral_records,
    resolve_experiment_root,
    select_canonical_pairs,
)
from wwgpt.publication_plots import PublicationPlotConfig, build_all_publication_figures

raw_results_root = Path(RESULTS_ROOT_SETTING).expanduser()
if not raw_results_root.is_absolute():
    candidates = [cwd / raw_results_root, repo / raw_results_root]
    existing = [p for p in candidates if p.exists()]
    raw_results_root = existing[0] if existing else candidates[0]

RESULTS_ROOT = resolve_experiment_root(raw_results_root)
ANALYSIS_DIR = RESULTS_ROOT / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {repo}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Analysis dir: {ANALYSIS_DIR}")
print(f"Data root: {DATA_ROOT}")

## Discover canonical runs

In [ ]:
candidates = discover_pair_candidates(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
canonical_pairs, pair_audit = select_canonical_pairs(candidates)
runs = discover_canonical_runs(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)

if not runs:
    raise RuntimeError(
        "No canonical completed runs were discovered. Set WWGPT_RESULTS_ROOT to a results tree "
        "containing explicit manifest-backed pair_* or trial_* runs; fixture data is not substituted automatically."
    )

missing_manifests = [str(r["run_dir"]) for r in runs if not (Path(r["run_dir"]) / "manifest.json").exists()]
if missing_manifests:
    raise RuntimeError(f"Required manifest.json files are missing: {missing_manifests}")

inventory = build_run_inventory(runs)
pair_audit.to_csv(ANALYSIS_DIR / "pair_candidate_audit.csv", index=False)
inventory.to_csv(ANALYSIS_DIR / "run_inventory.csv", index=False)
print(f"Canonical pairs: {len(canonical_pairs)}; canonical runs: {len(runs)}")
display(pair_audit)
display(inventory)

## Normalize records and validate WeightWatcher rows

In [ ]:
spectral_frames = []
metric_frames = []
projection_frames = []
unsupported_optional_fields = []

for r in runs:
    art = load_run_artifacts(Path(r["run_dir"]))
    context = {
        "seed": r["seed"],
        "pair_id": r["pair_id"],
        "optimizer_family": r["optimizer_family"],
        "optimizer_label": r["optimizer_label"],
        "optimizer_raw": r["optimizer_raw"],
        "run_dir": str(r["run_dir"]),
        "valid_for_science": bool(r["valid_for_science"]),
        "scientific_schema_version": r["manifest"].get("scientific_schema_version"),
    }
    spectral = normalize_spectral_records(art["spectral"]).assign(**context)
    metrics = normalize_metrics(art["metrics"]).assign(**context)
    projection = normalize_projection_records(art["projection"])
    spectral_frames.append(spectral)
    metric_frames.append(metrics)
    if not projection.empty:
        projection_frames.append(projection.assign(**context))

spectral_records = pd.concat(spectral_frames, ignore_index=True)
metric_records = pd.concat(metric_frames, ignore_index=True)
projection_records = pd.concat(projection_frames, ignore_index=True) if projection_frames else pd.DataFrame()

spectral_records.to_csv(ANALYSIS_DIR / "spectral_records_scientific.csv", index=False)
metric_records.to_csv(ANALYSIS_DIR / "metrics_records_scientific.csv", index=False)
projection_records.to_csv(ANALYSIS_DIR / "projection_records_scientific.csv", index=False)

optional_fields = ["D", "num_evals", "spectral_norm", "stable_rank", "log_alpha_norm", "alpha_weighted"]
for field in optional_fields:
    if field not in spectral_records or spectral_records[field].dropna().empty:
        unsupported_optional_fields.append({"field": field, "explanation": "not present or entirely missing in WeightWatcher output; optional plots using this field are skipped"})
pd.DataFrame(unsupported_optional_fields).to_csv(ANALYSIS_DIR / "unsupported_weightwatcher_fields.csv", index=False)

validity_audit = audit_spectral_validity(spectral_records)
validity_audit.to_csv(ANALYSIS_DIR / "spectral_validity_audit.csv", index=False)
if not validity_audit["valid_for_weightwatcher_science"].all():
    bad = validity_audit.loc[~validity_audit["valid_for_weightwatcher_science"], ["row_index", "invalid_reasons"]].head(10).to_dict("records")
    raise RuntimeError(f"Unverifiable spectral rows refused before plotting: {bad}")

ww_records = spectral_records.loc[validity_audit["valid_for_weightwatcher_science"].to_numpy()].copy()
print(f"Validated WeightWatcher rows: {len(ww_records)}")
display(validity_audit.groupby(["optimizer_family", "seed"], dropna=False).agg(rows=("row_index", "size"), valid_rows=("valid_for_weightwatcher_science", "sum")).reset_index())

## Scientific summaries

In [ ]:
projected = ww_records[ww_records["layer_group"].eq("projected_transformer_matrix")].copy()
if projected.empty:
    raise RuntimeError("No projected transformer matrix WeightWatcher rows are available for scientific summaries.")

for col in ["step", "tokens_seen", "alpha", "D", "num_evals", "spectral_norm", "stable_rank"]:
    if col in projected:
        projected[col] = pd.to_numeric(projected[col], errors="coerce")

per_layer_summary = (
    projected.groupby(["optimizer_family", "seed", "tokens_seen", "layer_name"], dropna=False)
    .agg(mean_alpha=("alpha", "mean"), median_alpha=("alpha", "median"), record_count=("alpha", "count"))
    .reset_index()
)
per_snapshot_summary = (
    per_layer_summary.groupby(["optimizer_family", "seed", "tokens_seen"], dropna=False)
    .agg(mean_alpha=("mean_alpha", "mean"), median_alpha=("median_alpha", "median"), layer_count=("layer_name", "nunique"))
    .reset_index()
)
per_layer_summary.to_csv(ANALYSIS_DIR / "projected_layer_alpha_summary.csv", index=False)
per_snapshot_summary.to_csv(ANALYSIS_DIR / "run_snapshot_alpha_summary.csv", index=False)
display(per_snapshot_summary.head())

## Publication figures

In [ ]:
plot_config = PublicationPlotConfig(band=PLOT_BAND, dpi=PUBLICATION_DPI, file_format=SOURCE_DATA_FORMAT)
for r in runs:
    r["artifacts"] = load_run_artifacts(Path(r["run_dir"]))
figure_outputs = build_all_publication_figures(runs, ANALYSIS_DIR / "publication_figures", plot_config)
figure_manifest = pd.DataFrame([{"figure": name, **{k: str(v) for k, v in paths.items() if v is not None}} for name, paths in figure_outputs.items()])
figure_manifest.to_csv(ANALYSIS_DIR / "weightwatcher_plot_manifest.csv", index=False)
display(figure_manifest)
print("Uncertainty bands aggregate within each run/snapshot first, then compute mean ± sample standard deviation or 95% Student-t confidence interval across independent seeds only.")